In [ ]:
import numpy as np
import pandas as pd
import math
import random
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn import ensemble  
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor 
from sklearn.neural_network import MLPRegressor 
from xgboost import XGBRegressor
import xgboost as xgb

In [ ]:
#using new timeseries dataset
# import copy
df = pd.read_csv(r"D:\FPL\fpl-optimization\point-predictor\code\xP Model\data\all_players_timeseries_dataset.csv", index_col=False)
train = df[~df['season_x'].isin(["2023-24", "2024-25"])]
test = df[df['season_x'].isin(["2023-24", "2024-25"])]


In [ ]:
train.shape

In [ ]:
# display all columns in train df without truncating columns
pd.set_option('display.max_columns', None)
train.head()

In [ ]:
train.iloc[1068:1072, 0:10]

In [ ]:
train.iloc[1068:1072, 10:20]

In [ ]:
train.iloc[1068:1072, 20:30]

In [ ]:
train.iloc[1068:1072, 30:40]

In [ ]:
def plot_nas(df: pd.DataFrame):
    if df.isnull().sum().sum() != 0:
        na_df = (df.isnull().sum() / len(df)) * 100      
        na_df = na_df.drop(na_df[na_df == 0].index).sort_values(ascending=False)
        missing_data = pd.DataFrame({'Missing Ratio %' :na_df})
        missing_data.plot(kind = "barh")
        plt.show()
    else:
        print('No NAs found')
plot_nas(train)
plot_width, plot_height = (20,24)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)

In [ ]:
train_clean = train.dropna()
train_clean = train_clean.reset_index(drop=True)

In [ ]:
plot_nas(train_clean)
plot_width, plot_height = (20,24)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)

In [ ]:
# Define a mapping of position names to numerical codes
position_mapping = {'GK': 1, 'GKP':1, 'DEF': 2, 'MID': 3, 'FWD': 4}

# Encode the "position" column using the mapping
train_clean['position_encoded'] = train_clean['position'].replace(position_mapping)


In [ ]:
train_clean.dtypes[train_clean.dtypes == 'object']

In [ ]:
# Use mapping of team names to numerical codes based on reverse order of PL League table total points
team_key_df = pd.read_csv(r"D:\FPL\fpl-optimization\point-predictor\code\xP Model\data\team_key.csv", index_col=False)
team_name_to_code = team_key_df.set_index('team')['team_code'].to_dict()

In [ ]:
team_name_to_code

In [ ]:


# # Use mapping of team names to numerical codes based on reverse order of PL League table total points
# team_key_df = pd.read_csv("datasets\\top_players_timeseries_dataset.csv", index_col=False)
# team_name_to_code = {team: code for code, team in enumerate(unique_teams, start=1)}

# Replace the team names in the "opponent" column with numerical codes
train_clean['team_x'] = train_clean['team_x'].replace(team_name_to_code)
train_clean['opponent'] = train_clean['opp_team_name'].replace(team_name_to_code)
train_clean['1_opponent'] = train_clean['1_opp_team_name'].replace(team_name_to_code)
train_clean['2_opponent'] = train_clean['2_opp_team_name'].replace(team_name_to_code)
train_clean['3_opponent'] = train_clean['3_opp_team_name'].replace(team_name_to_code)
train_clean['4_opponent'] = train_clean['4_opp_team_name'].replace(team_name_to_code)
# Save the key (mapping) to a CSV file
# team_key_df = pd.DataFrame(team_name_to_code.items(), columns=['team_name', 'team_code'])
# team_key_df.to_csv('team_key.csv', index=False)

# Display the updated dataset with the encoded "opponent" column
print(train_clean)

In [ ]:
train_clean.tail()

In [ ]:
train_clean = train_clean.drop(['position', 'opp_team_name', '1_opp_team_name', '2_opp_team_name','3_opp_team_name','4_opp_team_name'], axis=1)
train_clean['1_was_home'] = train_clean['1_was_home'].astype('bool')
train_clean['2_was_home'] = train_clean['2_was_home'].astype('bool')
train_clean['3_was_home'] = train_clean['3_was_home'].astype('bool')
train_clean['4_was_home'] = train_clean['4_was_home'].astype('bool')
train_clean.dtypes[train_clean.dtypes == 'object']

In [ ]:
train_clean.head()

In [ ]:
# combine columns such as fdr_team and fdr_opp_team as fdr_net as subtraction of two columns and transfers_in and transfers_out as transfers_net and drop the columns
train_clean['fdr_net'] = train_clean['fdr_team'] - train_clean['fdr_opp_team']
train_clean['transfers_net'] = train_clean['transfers_in'] - train_clean['transfers_out']
train_clean['1_transfers_net'] = train_clean['1_transfers_in'] - train_clean['1_transfers_out']
train_clean['2_transfers_net'] = train_clean['2_transfers_in'] - train_clean['2_transfers_out']
train_clean['3_transfers_net'] = train_clean['3_transfers_in'] - train_clean['3_transfers_out']
train_clean['4_transfers_net'] = train_clean['4_transfers_in'] - train_clean['4_transfers_out']
train_clean = train_clean.drop(['fdr_team', 'fdr_opp_team', 'transfers_in', 'transfers_out', '1_transfers_in', '1_transfers_out', '2_transfers_in', '2_transfers_out', '3_transfers_in', '3_transfers_out', '4_transfers_in', '4_transfers_out'], axis=1)

In [ ]:
X = train_clean.drop(['name','season_x','total_points'], axis=1)
y = train_clean['total_points']

In [ ]:
# X['1_was_home'] = X['1_was_home'].astype('bool')
X.dtypes[X.dtypes == 'object']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [ ]:
scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)   

In [ ]:
#corrplot

# Compute the correlation matrix
corr = train_clean.corr()
print(corr)
# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
fig, ax = plt.subplots(figsize=(40,40))

# Draw the heatmap with the mask and correct aspect ratio
vmax = np.abs(corr.values[~mask]).max()
sns.heatmap(corr, mask=mask, cmap=plt.cm.PuOr, vmin=-vmax, vmax=vmax,
            square=True, linecolor="lightgray", linewidths=1, ax=ax)
# for i in range(len(corr)):
#     ax.text(i+0.5,len(corr)-(i+0.5), corr.columns[i], 
#             ha="center", va="center", rotation=45)
#     for j in range(i+1, len(corr)):
#         s = "{:.3f}".format(corr.values[i,j])
#         ax.text(j+0.5,len(corr)-(i+0.5),s, 
#             ha="center", va="center")
# ax.axis("off")
plt.show()

In [ ]:
ts_points = train_clean[['total_points','1_total_points','2_total_points','3_total_points','4_total_points','1_ict_index','1_creativity','1_influence','1_threat','2_ict_index','2_creativity','2_influence','2_threat','3_ict_index','3_creativity','3_influence','3_threat','4_ict_index','4_creativity','4_influence','4_threat']]
corr = ts_points.corr()
print(corr)
# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
fig, ax = plt.subplots()

# Draw the heatmap with the mask and correct aspect ratio
vmax = np.abs(corr.values[~mask]).max()
sns.heatmap(corr, mask=mask, cmap=plt.cm.PuOr, vmin=-vmax, vmax=vmax,
            square=True, linecolor="lightgray", linewidths=1, ax=ax)
# for i in range(len(corr)):
#     ax.text(i+0.5,len(corr)-(i+0.5), corr.columns[i], 
#             ha="center", va="center", rotation=45)
#     for j in range(i+1, len(corr)):
#         s = "{:.3f}".format(corr.values[i,j])
#         ax.text(j+0.5,len(corr)-(i+0.5),s, 
#             ha="center", va="center")
# ax.axis("off")
plt.show()

In [ ]:
selected_points = train_clean[['total_points','1_selected','2_selected','3_selected','4_selected']]
corr = selected_points.corr()
print(corr)
# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
fig, ax = plt.subplots()

# Draw the heatmap with the mask and correct aspect ratio
vmax = np.abs(corr.values[~mask]).max()
sns.heatmap(corr, mask=mask, cmap=plt.cm.PuOr, vmin=-vmax, vmax=vmax,
            square=True, linecolor="lightgray", linewidths=1, ax=ax)
# for i in range(len(corr)):
#     ax.text(i+0.5,len(corr)-(i+0.5), corr.columns[i], 
#             ha="center", va="center", rotation=45)
#     for j in range(i+1, len(corr)):
#         s = "{:.3f}".format(corr.values[i,j])
#         ax.text(j+0.5,len(corr)-(i+0.5),s, 
#             ha="center", va="center")
# ax.axis("off")
plt.show()

In [ ]:
value_points = train_clean[['total_points','value','1_value','2_value','3_value','4_value','position_encoded']]
corr = value_points.corr()
print(corr)
# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
fig, ax = plt.subplots()

# Draw the heatmap with the mask and correct aspect ratio
vmax = np.abs(corr.values[~mask]).max()
sns.heatmap(corr, mask=mask, cmap=plt.cm.PuOr, vmin=-vmax, vmax=vmax,
            square=True, linecolor="lightgray", linewidths=1, ax=ax)
# for i in range(len(corr)):
#     ax.text(i+0.5,len(corr)-(i+0.5), corr.columns[i], 
#             ha="center", va="center", rotation=45)
#     for j in range(i+1, len(corr)):
#         s = "{:.3f}".format(corr.values[i,j])
#         ax.text(j+0.5,len(corr)-(i+0.5),s, 
#             ha="center", va="center")
# ax.axis("off")
plt.show()

In [ ]:
match_stats = train_clean[['total_points','position_encoded','team_x','1_opponent','1_minutes','1_goals_scored','1_assists','1_clean_sheets','1_goals_conceded','1_own_goals','1_saves','1_penalties_saved','1_penalties_missed','1_goals_scored_team','1_goals_conceded_team','1_red_cards','1_yellow_cards']]
corr = match_stats.corr()
print(corr)
# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
fig, ax = plt.subplots()

# Draw the heatmap with the mask and correct aspect ratio
vmax = np.abs(corr.values[~mask]).max()
sns.heatmap(corr, mask=mask, cmap=plt.cm.PuOr, vmin=-vmax, vmax=vmax,
            square=True, linecolor="lightgray", linewidths=1, ax=ax)
# for i in range(len(corr)):
#     ax.text(i+0.5,len(corr)-(i+0.5), corr.columns[i], 
#             ha="center", va="center", rotation=45)
#     for j in range(i+1, len(corr)):
#         s = "{:.3f}".format(corr.values[i,j])
#         ax.text(j+0.5,len(corr)-(i+0.5),s, 
#             ha="center", va="center")
# ax.axis("off")
plt.show()

In [ ]:
# match_stats = train_clean[['total_points','team_x','1_opponent','1_xG','1_xGA','1_npxG','1_npxGA','1_deep','1_deep_allowed','1_xpts','1_pts','1_opp_xpts','1_opp_pts']]
# corr = match_stats.corr()
# print(corr)
# # Generate a mask for the upper triangle
# mask = np.zeros_like(corr, dtype=np.bool)
# mask[np.triu_indices_from(mask)] = True

# # Set up the matplotlib figure
# fig, ax = plt.subplots(figsize=(20,20))

# # Draw the heatmap with the mask and correct aspect ratio
# vmax = np.abs(corr.values[~mask]).max()
# sns.heatmap(corr, mask=mask, cmap=plt.cm.PuOr, vmin=-vmax, vmax=vmax,
#             square=True, linecolor="lightgray", linewidths=1, ax=ax)
# # for i in range(len(corr)):
# #     ax.text(i+0.5,len(corr)-(i+0.5), corr.columns[i], 
# #             ha="center", va="center", rotation=45)
# #     for j in range(i+1, len(corr)):
# #         s = "{:.3f}".format(corr.values[i,j])
# #         ax.text(j+0.5,len(corr)-(i+0.5),s, 
# #             ha="center", va="center")
# # ax.axis("off")
# plt.show()

In [ ]:
# Baseline model
predict_train = np.mean(y_train)
# Get predictions on the test set
predict_test = np.ones(y_test.shape) * predict_train
residuals = predict_train - y_train
residuals2 = predict_test - y_test
mse_train = np.sqrt(sum(residuals**2)/len(residuals))
mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
abs_train = sum(abs(residuals))/len(residuals)
abs_test = sum(abs(residuals2))/len(residuals2)
print('sqrt(MSE) on train set: ', mse_train)
print('sqrt(MSE) on test set: ', mse_test)
print('Mean Absolute value of residuals on train set: ', abs_train)
print('Mean Absolute value of residuals on test set: ', abs_test)

In [ ]:
Regression = LinearRegression()

In [ ]:
Regression.fit(X_train, y_train)

In [ ]:
predict_train = Regression.predict(X_train)

predict_test = Regression.predict(X_test)

In [ ]:
plt.figure()
sns.scatterplot(np.arange(len(y_train)), predict_train)
plt.title('Predicted values for train set')

plt.figure()
sns.scatterplot(np.arange(len(y_test)), predict_test)
plt.title('Predicted values for test set')

In [ ]:
residuals = predict_train - y_train
plt.figure()
sns.scatterplot(np.arange(len(y_train)), residuals)
plt.ylabel('Residuals: y_hat - y')
plt.title('Plot of residuals for  train set')
residuals2 = predict_test - y_test
plt.figure()
sns.scatterplot(np.arange(len(y_test)), residuals2)
plt.ylabel('Residuals: y_hat - y')
plt.title('Plot of residuals for test set')

In [ ]:
alpha = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000, 1000000,10000000, 100000000]
mse_train = [math.nan for i in range(12)]
mse_test = [math.nan for i in range(12)]
abs_train = [math.nan for i in range(12)]
abs_test = [math.nan for i in range(12)]
for i in range(12):
    Regression = Ridge(alpha = alpha[i],fit_intercept=True)
    Regression.fit(X_train, y_train)
    
    predict_train = Regression.predict(X_train)

    predict_test = Regression.predict(X_test)


    residuals = predict_train - y_train

    residuals2 = predict_test - y_test
    
    mse_train[i] = np.sqrt(sum(residuals**2)/len(residuals))
    mse_test[i] = np.sqrt(sum(residuals2**2)/len(residuals2))
    
    abs_train[i] = sum(abs(residuals))/len(residuals)
    abs_test[i] = sum(abs(residuals2))/len(residuals2)

In [ ]:
plt.figure()
plt.plot(np.arange(-3, -3 + len(mse_train)), mse_train, label='Training set')
plt.plot(np.arange(-3, -3 +len(mse_test)), mse_test, label='Test set')
plt.xlabel('Order of magnitude of penalty term')
plt.ylabel('sqrt(MSE)')
plt.title('Performance on train and test set for different penalties')
plt.legend()
plt.show()

In [ ]:
print('Unnormalized penalty alpha: ', alpha[5])
print('Normalized penalty alpha: ', alpha[5]/len(X_train))

In [ ]:
i = 5
Regression = Ridge(alpha = alpha[i],fit_intercept=True, normalize=False)
Regression.fit(X_train, y_train)
predict_train = Regression.predict(X_train)
predict_test = Regression.predict(X_test)

In [ ]:
Regression.coef_

In [ ]:
residuals = predict_train - y_train
residuals2 = predict_test - y_test
mse_train = np.sqrt(sum(residuals**2)/len(residuals))
mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
abs_train = sum(abs(residuals))/len(residuals)
abs_test = sum(abs(residuals2))/len(residuals2)

In [ ]:
print('sqrt(MSE) on train set: ', mse_train)
print('sqrt(MSE) on test set: ', mse_test)
print('Mean Absolute value of residuals on train set: ', abs_train)
print('Mean Absolute value of residuals on test set: ', abs_test)

In [ ]:
plt.figure()
sns.scatterplot(np.arange(len(y_train)), predict_train)
plt.title('Predicted values for train set')

plt.figure()
sns.scatterplot(np.arange(len(y_test)), predict_test)
plt.title('Predicted values for test set')

In [ ]:
plt.figure()
sns.scatterplot(np.arange(len(y_train)), residuals)
plt.ylabel('Residuals: y_hat - y')
plt.title('Plot of residuals for  train set')

plt.figure()
sns.scatterplot(np.arange(len(y_test)), residuals2)
plt.ylabel('Residuals: y_hat - y')
plt.title('Plot of residuals for test set')

In [ ]:
plt.figure()
sns.scatterplot(y_train, residuals)
plt.ylabel('Residuals: y_hat - y')
plt.title('Plot of residuals for  train set')

plt.figure()
sns.scatterplot(y_test, residuals2)
plt.ylabel('Residuals: y_hat - y')
plt.title('Plot of residuals for test set')

In [ ]:

predict = Regression.predict(X_predict) #need to revise or bring down during test
residuals3 = predict - test.total_points
mse_predict = np.sqrt(sum(residuals3**2)/len(residuals3))
abs_predict = sum(abs(residuals3))/len(residuals3)
print('sqrt(MSE) on predict set: ', mse_predict)
print('Mean Absolute value of residuals on predict set: ', abs_predict)


Random Forest

In [ ]:
regr = RandomForestRegressor(oob_score = True, n_estimators = 100, max_features = 5)

regr.fit(X_train, y_train)
predict_train = regr.predict(X_train)

predict_test = regr.predict(X_test)

In [ ]:
out_of_bag_predict = regr.oob_score

In [ ]:
residuals = predict_train - y_train
residuals2 = predict_test - y_test
residuals3 = out_of_bag_predict - y_train
mse_train = np.sqrt(sum(residuals**2)/len(residuals))
mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
mse_out_of_bag = np.sqrt(sum(residuals3**2)/len(residuals3))
abs_train = sum(abs(residuals))/len(residuals)
abs_test = sum(abs(residuals2))/len(residuals2)
abs_out_of_bag = sum(abs(residuals3))/len(residuals3)

In [ ]:
print('sqrt(MSE) on train set: ', mse_train)
print('sqrt(MSE) on test set: ', mse_test)
print('sqrt(MSE) on out-of-bag set: ', mse_out_of_bag)
print('Mean Absolute value of residuals on train set: ', abs_train)
print('Mean Absolute value of residuals on test set: ', abs_test)
print('Mean Absolute value of residuals on out-of-bag set: ', abs_out_of_bag)

In [ ]:
importances = regr.feature_importances_

In [ ]:
sorted_indices = np.argsort(importances)[::-1]
 
feat_labels = X_train.columns
 
for f in range(X_train.shape[1]):
    print("%2d) %-*s %f" % (f + 1, 30,
                            feat_labels[sorted_indices[f]],
                            importances[sorted_indices[f]]))

In [ ]:
plt.figure()
plt.title('Feature Importance')
plot_width, plot_height = (30,24)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)
plt.bar(range(X_train.shape[1]), importances[sorted_indices], align='center')
plt.xticks(range(X_train.shape[1]), X_train.columns[sorted_indices], rotation=90)
plt.ylabel('Average decrease in MSE');
plt.tight_layout()
plt.show()

In [ ]:
X_train.shape[1]

In [ ]:
N = [1,5,10,25,50,100,110]
# N = len(features)
mse_train = [math.nan for i in range(len(N))]
mse_test = [math.nan for i in range(len(N))]
mse_out_of_bag = [math.nan for i in range(len(N))]
abs_train = [math.nan for i in range(len(N))]
abs_test = [math.nan for i in range(len(N))]
abs_out_of_bag = [math.nan for i in range(len(N))]


# for i in range(n):
#     Forest = RandomForestRegressor(oob_score = True, n_estimators = 100, max_features = features[i])
#     Forest.fit(X_train, y_train)
    
#     predict_train = Forest.predict(X_train)

#     predict_test = Forest.predict(X_test)


#     residuals = predict_train - y_train
#     residuals2 = predict_test - y_test
#     residuals3 = out_of_bag_predict - y_train
#     mse_train[i] = np.sqrt(sum(residuals**2)/len(residuals))
#     mse_test[i] = np.sqrt(sum(residuals2**2)/len(residuals2))
#     mse_out_of_bag[i] = np.sqrt(sum(residuals3**2)/len(residuals3))
#     abs_train[i] = sum(abs(residuals))/len(residuals)
#     abs_test[i] = sum(abs(residuals2))/len(residuals2)
#     abs_out_of_bag[i] = sum(abs(residuals3))/len(residuals3)
for i, n in enumerate(N):
    # Select the top n features based on importance
    top_n_features = np.argsort(importances)[::-1][:n]
    X_train_top_n = np.take(X_train, top_n_features, axis=1)
    X_test_top_n = np.take(X_test, top_n_features, axis=1)

    # Train a new Random Forest Regressor with the top n features
    rf_top_n = RandomForestRegressor(oob_score=True, n_estimators=100)
    rf_top_n.fit(X_train_top_n, y_train)

    # Make predictions on the training and testing sets
    predict_train = rf_top_n.predict(X_train_top_n)
    predict_test = rf_top_n.predict(X_test_top_n)

    # Calculate the residuals
    residuals = predict_train - y_train
    residuals2 = predict_test - y_test
    residuals3 = rf_top_n.oob_prediction_ - y_train

    # Calculate the MSE and MAE
    mse_train[i] = np.sqrt(sum(residuals**2)/len(residuals))
    mse_test[i] = np.sqrt(sum(residuals2**2)/len(residuals2))
    mse_out_of_bag[i] = np.sqrt(sum(residuals3**2)/len(residuals3))
    abs_train[i] = sum(abs(residuals))/len(residuals)
    abs_test[i] = sum(abs(residuals2))/len(residuals2)
    abs_out_of_bag[i] = sum(abs(residuals3))/len(residuals3)


In [ ]:
# Create a DataFrame to store the results
results = pd.DataFrame({'MSE_train': mse_train, 'MSE_test': mse_test, 'MSE_out_of_bag': mse_out_of_bag, 'MAE_train':abs_train,'MAE_test':abs_test,'MAE_out_of_bag':abs_out_of_bag}, index=N)

# Print the results as a table
print(results)

In [ ]:
plt.figure()
plot_width, plot_height = (6,4)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)
plt.plot(np.arange(7), mse_train, label="mse-train")
plt.plot(np.arange(7), mse_test, label="mse-test")
plt.plot(np.arange(7), mse_out_of_bag, label="mse-out of bag")
plt.title('number of features = [1,5,10,25,50,100,110]')
plt.legend(loc="lower right")


In [ ]:
# Select top 50 features by importance
top_50_features = np.argsort(importances)[::-1][:50]

# Select top 50 features from the data
X_train_selected = np.take(X_train, top_50_features, axis=1)
X_test_selected = np.take(X_test, top_50_features, axis=1)

depth = [5, 10, 20, 50, 100, 200]
n = len(depth)
mse_train = [math.nan for i in range(n)]
mse_test = [math.nan for i in range(n)]
mse_out_of_bag = [math.nan for i in range(n)]
abs_train = [math.nan for i in range(n)]
abs_test = [math.nan for i in range(n)]
abs_out_of_bag = [math.nan for i in range(n)]

for i in range(n):
    Forest = RandomForestRegressor(oob_score=True, n_estimators=100, max_features=None, max_depth=depth[i])
    Forest.fit(X_train_selected, y_train)
    
    predict_train = Forest.predict(X_train_selected)
    predict_test = Forest.predict(X_test_selected)
    
    residuals = predict_train - y_train
    residuals2 = predict_test - y_test
    residuals3 = Forest.oob_prediction_ - y_train
    mse_train[i] = np.sqrt(sum(residuals**2)/len(residuals))
    mse_test[i] = np.sqrt(sum(residuals2**2)/len(residuals2))
    mse_out_of_bag[i] = np.sqrt(sum(residuals3**2)/len(residuals3))
    abs_train[i] = sum(abs(residuals))/len(residuals)
    abs_test[i] = sum(abs(residuals2))/len(residuals2)
    abs_out_of_bag[i] = sum(abs(residuals3))/len(residuals3)

In [ ]:
# Create a DataFrame to store the results
results = pd.DataFrame({'MSE_train': mse_train, 'MSE_test': mse_test, 'MSE_out_of_bag': mse_out_of_bag, 'MAE_train':abs_train,'MAE_test':abs_test,'MAE_out_of_bag':abs_out_of_bag}, index=depth)

# Print the results as a table
print(results)

In [ ]:
plt.figure()
plot_width, plot_height = (6,4)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)
plt.plot(depth, mse_train, label="mse-train")
plt.plot(depth, mse_test, label="mse-test")
plt.plot(depth, mse_out_of_bag, label="mse-out-of-bag")
plt.xlabel('Max Tree Depth')
plt.ylabel('sqrt(MSE)')
plt.title('sqrt(MSE) for oob, test and train')
plt.legend(loc="lower right")

In [ ]:
# rf_final = RandomForestRegressor(oob_score = True, n_estimators = 100, max_features = 25, max_depth=40)

# rf_final.fit(X_train, y_train)
# predict_train = rf_final.predict(X_train)

# predict_test = rf_final.predict(X_test)
# out_of_bag_predict = rf_final.oob_score

# residuals = predict_train - y_train
# residuals2 = predict_test - y_test
# residuals3 = out_of_bag_predict - y_train
# mse_train = np.sqrt(sum(residuals**2)/len(residuals))
# mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
# mse_out_of_bag = np.sqrt(sum(residuals3**2)/len(residuals3))
# abs_train = sum(abs(residuals))/len(residuals)
# abs_test = sum(abs(residuals2))/len(residuals2)
# abs_out_of_bag = sum(abs(residuals3))/len(residuals3)

# print('sqrt(MSE) on train set: ', mse_train)
# print('sqrt(MSE) on test set: ', mse_test)
# print('sqrt(MSE) on oob set: ', mse_out_of_bag)
# print('Mean Absolute value of residuals on train set: ', abs_train)
# print('Mean Absolute value of residuals on test set: ', abs_test)
# print('Mean Absolute value of residuals on oob set: ', abs_out_of_bag)

In [ ]:
# Select top 50 features by importance
top_50_features = np.argsort(importances)[::-1][:50]

# Select top 50 features from the data
X_train_selected = np.take(X_train, top_50_features, axis=1)
X_test_selected = np.take(X_test, top_50_features, axis=1)

estimators = [1,10,100,1000,10000]
n = len(estimators)
mse_train = [math.nan for i in range(n)]
mse_test = [math.nan for i in range(n)]
mse_out_of_bag = [math.nan for i in range(n)]
abs_train = [math.nan for i in range(n)]
abs_test = [math.nan for i in range(n)]
abs_out_of_bag = [math.nan for i in range(n)]

for i in range(n):
    Forest = RandomForestRegressor(oob_score=True, n_estimators=estimators[i], max_features=None, max_depth=10) #selecting depth=10
    Forest.fit(X_train_selected, y_train)
    
    predict_train = Forest.predict(X_train_selected)
    predict_test = Forest.predict(X_test_selected)
    
    residuals = predict_train - y_train
    residuals2 = predict_test - y_test
    residuals3 = Forest.oob_prediction_ - y_train
    mse_train[i] = np.sqrt(sum(residuals**2)/len(residuals))
    mse_test[i] = np.sqrt(sum(residuals2**2)/len(residuals2))
    mse_out_of_bag[i] = np.sqrt(sum(residuals3**2)/len(residuals3))
    abs_train[i] = sum(abs(residuals))/len(residuals)
    abs_test[i] = sum(abs(residuals2))/len(residuals2)
    abs_out_of_bag[i] = sum(abs(residuals3))/len(residuals3)

In [ ]:
# Create a DataFrame to store the results
results = pd.DataFrame({'MSE_train': mse_train, 'MSE_test': mse_test, 'MSE_out_of_bag': mse_out_of_bag, 'MAE_train':abs_train,'MAE_test':abs_test,'MAE_out_of_bag':abs_out_of_bag}, index=estimators)

# Print the results as a table
print(results)

In [ ]:
plt.figure()
plot_width, plot_height = (6,4)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)
plt.plot(np.arange(0, len(mse_train)), mse_train, label="mse-train")
plt.plot(np.arange(0, len(mse_train)), mse_test, label="mse-test")
plt.plot(np.arange(0, len(mse_train)), mse_out_of_bag, label="mse-out-of-bag")
plt.xlabel('Order of magnitude of estimators')
plt.ylabel('sqrt(MSE)')
plt.title('sqrt(MSE) for oob, test and train')
plt.legend(loc="lower right")

In [ ]:
# Select top 50 features by importance
top_50_features = np.argsort(importances)[::-1][:50]

# Select top 50 features from the data
X_train_selected = np.take(X_train, top_50_features, axis=1)
X_test_selected = np.take(X_test, top_50_features, axis=1)

# Final parameter - features = 50, depth = 10, estimators = 100
rf_final = RandomForestRegressor(oob_score = True, n_estimators = 100, max_features = None, max_depth=10)

rf_final.fit(X_train_selected, y_train)
predict_train = rf_final.predict(X_train_selected)

predict_test = rf_final.predict(X_test_selected)
out_of_bag_predict = rf_final.oob_score

residuals = predict_train - y_train
residuals2 = predict_test - y_test
residuals3 = out_of_bag_predict - y_train
mse_train = np.sqrt(sum(residuals**2)/len(residuals))
mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
mse_out_of_bag = np.sqrt(sum(residuals3**2)/len(residuals3))
abs_train = sum(abs(residuals))/len(residuals)
abs_test = sum(abs(residuals2))/len(residuals2)
abs_out_of_bag = sum(abs(residuals3))/len(residuals3)

print('sqrt(MSE) on train set: ', mse_train)
print('sqrt(MSE) on test set: ', mse_test)
print('sqrt(MSE) on oob set: ', mse_out_of_bag)
print('Mean Absolute value of residuals on train set: ', abs_train)
print('Mean Absolute value of residuals on test set: ', abs_test)
print('Mean Absolute value of residuals on oob set: ', abs_out_of_bag)

In [ ]:
filename = 'rf_model.sav'
pickle.dump(rf_final, open(filename, 'wb'))

In [ ]:
# test = pd.read_csv("test_dataset.csv")

In [ ]:
test.head()

In [ ]:
plot_nas(test)
plot_width, plot_height = (20,24)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)

In [ ]:
test_clean = test.dropna()
# test_clean = test_clean.reset_index(drop=True)

In [ ]:
X_train.head()

In [ ]:
# Encode the "position" column using the mapping
test_clean['position_encoded'] = test_clean['position'].replace(position_mapping)

In [ ]:
test_clean.dtypes[test_clean.dtypes == 'object']

In [ ]:
# # Extract unique team names from the "team" column
# unique_teams = train_clean['team_name'].unique()

# Create a mapping of team names to numerical codes
# team_name_to_code = {team: code for code, team in enumerate(unique_teams, start=1)}

# Replace the team names in the "opponent" column with numerical codes
test_clean['team_x'] = test_clean['team_x'].replace(team_name_to_code)
test_clean['opponent'] = test_clean['opp_team_name'].replace(team_name_to_code)
test_clean['1_opponent'] = test_clean['1_opp_team_name'].replace(team_name_to_code)
test_clean['2_opponent'] = test_clean['2_opp_team_name'].replace(team_name_to_code)
test_clean['3_opponent'] = test_clean['3_opp_team_name'].replace(team_name_to_code)
test_clean['4_opponent'] = test_clean['4_opp_team_name'].replace(team_name_to_code)
# Save the key (mapping) to a CSV file
# team_key_df = pd.DataFrame(team_name_to_code.items(), columns=['team_name', 'team_code'])
# team_key_df.to_csv('team_key.csv', index=False)

# Display the updated dataset with the encoded "opponent" column
print(test_clean)

In [ ]:
test_clean = test_clean.drop(['position', 'opp_team_name', '1_opp_team_name', '2_opp_team_name','3_opp_team_name','4_opp_team_name'], axis=1)
test_clean['1_was_home'] = test_clean['1_was_home'].astype('bool')
test_clean['2_was_home'] = test_clean['2_was_home'].astype('bool')
test_clean['3_was_home'] = test_clean['3_was_home'].astype('bool')
test_clean['4_was_home'] = test_clean['4_was_home'].astype('bool')
test_clean.dtypes[test_clean.dtypes == 'object']

In [ ]:
test_clean.head(15)

In [ ]:
# combine columns such as fdr_team and fdr_opp_team as fdr_net as subtraction of two columns and transfers_in and transfers_out as transfers_net and drop the columns
test_clean['fdr_net'] = test_clean['fdr_team'] - test_clean['fdr_opp_team']
test_clean['transfers_net'] = test_clean['transfers_in'] - test_clean['transfers_out']
test_clean['1_transfers_net'] = test_clean['1_transfers_in'] - test_clean['1_transfers_out']
test_clean['2_transfers_net'] = test_clean['2_transfers_in'] - test_clean['2_transfers_out']
test_clean['3_transfers_net'] = test_clean['3_transfers_in'] - test_clean['3_transfers_out']
test_clean['4_transfers_net'] = test_clean['4_transfers_in'] - test_clean['4_transfers_out']
test_clean = test_clean.drop(['fdr_team', 'fdr_opp_team', 'transfers_in', 'transfers_out', '1_transfers_in', '1_transfers_out', '2_transfers_in', '2_transfers_out', '3_transfers_in', '3_transfers_out', '4_transfers_in', '4_transfers_out'], axis=1)

In [ ]:
X_predict = test_clean.drop(['name','season_x','total_points'], axis=1)

In [ ]:
X_predict.columns

In [ ]:
X_train.columns

In [ ]:
# Select top 50 features by importance
top_50_features = np.argsort(importances)[::-1][:50]

# Select top 50 features from the data
X_predict_selected = np.take(X_predict, top_50_features, axis=1)
Y_predict = rf_final.predict(X_predict_selected)

In [ ]:
Y_predict

In [ ]:
# test_clean.reset_index(level=0, inplace=True)

In [ ]:
test_clean['xP_rf'] = Y_predict

In [ ]:
test_clean.head()

In [ ]:
residuals = test_clean['xP_rf'] - test_clean['total_points']

mse_predict = np.sqrt(sum(residuals**2)/len(residuals))

abs_predict = sum(abs(residuals))/len(residuals)


print('sqrt(MSE) on prediction set: ', mse_predict)

print('Mean Absolute value of residuals on prediction set: ', abs_predict)

In [ ]:
test_clean.to_csv(r'data/test_predictions.csv', index=False)

In [ ]:
# Y_predict = rf_final.predict(X)
# train_clean['prediction'] = Y_predict
# train_clean.to_csv('train_predictions.csv', index=False)

In [ ]:
xgb = XGBRegressor(n_estimators=100, max_depth=7, eta=0.1, subsample=0.7, colsample_bytree=0.8)
xgb.fit(X_train, y_train)
predict_train = xgb.predict(X_train)

predict_test = xgb.predict(X_test)
# out_of_bag_predict = rf_final.oob_score

residuals = predict_train - y_train
residuals2 = predict_test - y_test
# residuals3 = out_of_bag_predict - y_train
mse_train = np.sqrt(sum(residuals**2)/len(residuals))
mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
# mse_out_of_bag = np.sqrt(sum(residuals3**2)/len(residuals3))
abs_train = sum(abs(residuals))/len(residuals)
abs_test = sum(abs(residuals2))/len(residuals2)
# abs_out_of_bag = sum(abs(residuals3))/len(residuals3)

print('sqrt(MSE) on train set: ', mse_train)
print('sqrt(MSE) on test set: ', mse_test)
# print('sqrt(MSE) on oob set: ', mse_out_of_bag)
print('Mean Absolute value of residuals on train set: ', abs_train)
print('Mean Absolute value of residuals on test set: ', abs_test)
# print('Mean Absolute value of residuals on oob set: ', abs_out_of_bag)

In [ ]:
# Get feature importance values
importances = xgb.feature_importances_

# Sort feature importance in descending order
sorted_importances = sorted(zip(X_train.columns, importances), key=lambda x: x[1], reverse=True)

# Plot feature importance
plt.bar([x[0] for x in sorted_importances], [x[1] for x in sorted_importances])
plt.xlabel('Feature Name')
plt.ylabel('Importance')
plt.xticks(rotation=90)  # rotate x-axis labels for better readability
plt.show()

In [ ]:
depth = range(1,15)
n = len(depth)
mse_train = [math.nan for i in range(n)]
mse_test = [math.nan for i in range(n)]
mse_out_of_bag = [math.nan for i in range(n)]
abs_train = [math.nan for i in range(n)]
abs_test = [math.nan for i in range(n)]
abs_out_of_bag = [math.nan for i in range(n)]

for i in range(n):
    xgb = XGBRegressor(n_estimators=100, max_depth=depth[i], eta=0.1, subsample=0.7, colsample_bytree=0.8)
    xgb.fit(X_train, y_train)
    
    predict_train = xgb.predict(X_train)

    predict_test = xgb.predict(X_test)    
    
    residuals = predict_train - y_train
    residuals2 = predict_test - y_test
    residuals3 = out_of_bag_predict - y_train
    mse_train[i] = np.sqrt(sum(residuals**2)/len(residuals))
    mse_test[i] = np.sqrt(sum(residuals2**2)/len(residuals2))
#     mse_out_of_bag[i] = np.sqrt(sum(residuals3**2)/len(residuals3))
    abs_train[i] = sum(abs(residuals))/len(residuals)
    abs_test[i] = sum(abs(residuals2))/len(residuals2)
#     abs_out_of_bag[i] = sum(abs(residuals3))/len(residuals3)

In [ ]:
mse_test

In [ ]:
plt.figure()
plot_width, plot_height = (6,4)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)
plt.plot(depth, mse_train, label="mse-train")
plt.plot(depth, mse_test, label="mse-test")
# plt.plot(depth, mse_out_of_bag, label="mse-out-of-bag")
plt.xlabel('Max Tree Depth')
plt.ylabel('sqrt(MSE)')
plt.title('sqrt(MSE) for test and train')
plt.legend(loc="lower right")

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
params = {
    # Parameters to tune.
    'max_depth':6,
    'min_child_weight': 1,
    'eta':.3,
    'subsample': 1,
    'colsample_bytree': 1,
    # Other parameters
    'objective':'reg:linear',
}


In [ ]:
params['eval_metric'] = "rmse"
num_boost_round = 999

xgb1 = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=10
)

In [ ]:
print("Best RMSE: {:.2f} with {} rounds".format(
                 xgb1.best_score,
                 xgb1.best_iteration+1))

In [ ]:
cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    seed=42,
    nfold=5,
    metrics={'rmse'},
    early_stopping_rounds=10
)
cv_results

In [ ]:
cv_results['test-rmse-mean'].min()

In [ ]:
gridsearch_params = [
    (max_depth, min_child_weight)
    for max_depth in range(9,12)
    for min_child_weight in range(5,8)
]

In [ ]:
min_rmse = float("Inf")
best_params = None
for max_depth, min_child_weight in gridsearch_params:
    print("CV with max_depth={}, min_child_weight={}".format(
                             max_depth,
                             min_child_weight))
    # Update our parameters
    params['max_depth'] = max_depth
    params['min_child_weight'] = min_child_weight
    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'rmse'},
        early_stopping_rounds=10
    )
    # Update best MAE
    mean_rmse = cv_results['test-rmse-mean'].min()
    boost_rounds = cv_results['test-rmse-mean'].argmin()
    print("\tRMSE {} for {} rounds".format(mean_rmse, boost_rounds))
    if mean_rmse < min_rmse:
        min_rmse = mean_rmse
        best_params = (max_depth,min_child_weight)
print("Best params: {}, {}, RMSE: {}".format(best_params[0], best_params[1], min_rmse))

In [ ]:
params['max_depth'] = 9
params['min_child_weight'] = 7

In [ ]:
gridsearch_params = [
    (subsample, colsample)
    for subsample in [i/10. for i in range(7,11)]
    for colsample in [i/10. for i in range(7,11)]
]

In [ ]:
min_rmse = float("Inf")
best_params = None
# We start by the largest values and go down to the smallest
for subsample, colsample in reversed(gridsearch_params):
    print("CV with subsample={}, colsample={}".format(
                             subsample,
                             colsample))
    # We update our parameters
    params['subsample'] = subsample
    params['colsample_bytree'] = colsample
    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'rmse'},
        early_stopping_rounds=10
    )
    # Update best score
    mean_rmse = cv_results['test-rmse-mean'].min()
    boost_rounds = cv_results['test-rmse-mean'].argmin()
    print("\tRMSE {} for {} rounds".format(mean_rmse, boost_rounds))
    if mean_rmse < min_rmse:
        min_rmse = mean_rmse
        best_params = (subsample,colsample)
print("Best params: {}, {}, RMSE: {}".format(best_params[0], best_params[1], min_rmse))

In [ ]:
params['subsample'] = 0.8
params['colsample_bytree'] = 1.0

In [ ]:
min_rmse = float("Inf")
best_params = None
for eta in [.3, .2, .1, .05, .01, .005]:
    print("CV with eta={}".format(eta))
    # We update our parameters
    params['eta'] = eta
    # Run and time CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics=['rmse'],
        early_stopping_rounds=10
      )
    # Update best score
    mean_rmse = cv_results['test-rmse-mean'].min()
    boost_rounds = cv_results['test-rmse-mean'].argmin()
    print("\tRMSE {} for {} rounds".format(mean_rmse, boost_rounds))
    if mean_rmse < min_rmse:
        min_rmse = mean_rmse
        best_params = eta
print("Best params: {}, RMSE: {}".format(best_params, min_rmse))

In [ ]:
params['eta'] = .01

In [ ]:
xgb_final = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=10
)

In [ ]:
num_boost_round = xgb_final.best_iteration + 1
xgb_best = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")]
)

In [ ]:
num_boost_round = xgb_final.best_iteration + 1
xgb_best = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtrain, "Train")]
)

In [ ]:
params

In [ ]:
filename = 'xgb_model.sav'
pickle.dump(xgb_best, open(filename, 'wb'))

In [ ]:
dpredict = xgb.DMatrix(X_predict, label=test_clean.total_points)
Y_xgb = xgb_best.predict(dpredict)

In [ ]:
Y_xgb.shape

In [ ]:
test_clean['xP_xgb'] = Y_xgb

In [ ]:
residuals2 = test_clean['xP_xgb'] - test_clean['total_points']

mse_predict2 = np.sqrt(sum(residuals2**2)/len(residuals2))

abs_predict2 = sum(abs(residuals2))/len(residuals2)


print('sqrt(MSE) on prediction set: ', mse_predict2)

print('Mean Absolute value of residuals on prediction set: ', abs_predict2)

In [ ]:
test_clean.to_csv('datasets\\predictions.csv', index=False)